In [13]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, mean_absolute_error
import xgboost as xgb
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import torch
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

In [14]:
core_stocks = ["aapl", "msft", "googl", "amzn", "nvda", "jpm", "bac", "gs", "xom", "gld", "jnj", "ko"]  
data_path = "Data_clean/Stocks clean"
sp500_path = "Data_clean/ETF clean/spy_clean.csv"
combined_df = pd.DataFrame()

In [15]:
for ticker in core_stocks:
    file_path = os.path.join(data_path, f"{ticker}_clean.csv")
    temp_df = pd.read_csv(file_path, index_col = "Date", parse_dates=True)
    combined_df[ticker] = temp_df["Close"]

sp500_df = pd.read_csv(sp500_path, index_col = "Date", parse_dates = True)

df = combined_df.copy()
df["spy"] = sp500_df["Close"]



In [16]:
df = df.sort_index()
start_date = '2005-02-25'
df = df.loc[start_date:]
df.head()

,aapl,msft,googl,amzn,nvda,jpm,bac,gs,xom,gld,jnj,ko,spy
Date,,,,,,,,,,,,,
2005-02-25,5.6975,21.173,92.935,34.99,8.9161,30.812,44.109,100.420,50.892,43.50,53.894,17.546,105.79
2005-02-28,5.7449,21.099,93.995,35.18,8.9625,30.463,43.980,99.241,50.937,43.52,53.390,17.456,105.08
2005-03-01,5.6987,21.199,93.030,35.39,8.9439,30.911,44.384,100.350,49.964,43.22,54.244,17.649,105.62
2005-03-02,5.6499,21.182,92.590,35.50,8.6843,30.828,43.904,100.090,50.433,43.25,54.496,17.555,105.57
2005-03-03,5.3518,21.109,93.505,35.65,8.5635,30.847,43.680,99.790,50.735,42.97,54.324,17.611,105.61


In [17]:
returns_df = df[core_stocks].pct_change()
returns_df['spy'] = df['spy'].pct_change().shift(-1)
returns_df = returns_df.loc["2005-02-28":]
returns_df = returns_df.dropna()
returns_df.head()

,aapl,msft,googl,amzn,nvda,jpm,bac,gs,xom,gld,jnj,ko,spy
Date,,,,,,,,,,,,,
2005-02-28,0.008319,-0.003495,0.011406,0.005430,0.005204,-0.011327,-0.002925,-0.011741,0.000884,0.000460,-0.009352,-0.005129,0.005139
2005-03-01,-0.008042,0.004740,-0.010267,0.005969,-0.002075,0.014706,0.009186,0.011175,-0.019102,-0.006893,0.015996,0.011056,-0.000473
2005-03-02,-0.008563,-0.000802,-0.004730,0.003108,-0.029025,-0.002685,-0.010815,-0.002591,0.009387,0.000694,0.004646,-0.005326,0.000379
2005-03-03,-0.052762,-0.003446,0.009882,0.004225,-0.013910,0.000616,-0.005102,-0.002997,0.005988,-0.006474,-0.003156,0.003190,0.012499
2005-03-04,0.024384,0.000000,-0.005936,0.005610,-0.004356,0.013453,0.009615,0.019341,0.008121,0.009542,0.014837,0.010732,0.000374


In [18]:
X = returns_df[core_stocks]
y = returns_df["spy"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, shuffle = False)

In [19]:
y_train_cls = (y_train > 0).astype(int)
y_test_cls = (y_test > 0).astype(int)

model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
model.fit(X_train, y_train_cls)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test_cls, y_pred)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.5309


In [20]:
y_train_cls = (y_train > 0).astype(int)
y_test_cls = (y_test > 0).astype(int)

model = lgb.LGBMClassifier(n_estimators = 100, learning_rate = 0.05, num_leaves = 8, max_depth = 3, random_state = 42, verbose = -1)
model.fit(X_train, y_train_cls)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test_cls, y_pred)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.5324


In [21]:
baseline_accuracy = y_test_cls.mean()
print(f"Accuracy (always up): {baseline_accuracy:.4f}")

Accuracy (always up): 0.5354


In [22]:
SEQ_LEN = 10 

X_values = X.values  
y_values = y.values

X_seq = []
y_seq = []

for i in range(len(X_values) - SEQ_LEN):
    window_X = X_values[i : i + SEQ_LEN]
    target_y = y_values[i + SEQ_LEN]
    
    X_seq.append(window_X)
    y_seq.append(target_y)

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(
    X_seq, y_seq, 
    test_size=0.2, 
    shuffle=False 
)

In [23]:
X_train_t = torch.tensor(X_train_seq, dtype=torch.float32)
y_train_t = torch.tensor(y_train_seq, dtype=torch.float32)
X_test_t = torch.tensor(X_test_seq, dtype=torch.float32)
y_test_t = torch.tensor(y_test_seq, dtype=torch.float32)

BATCH_SIZE = 32

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False) 

test_dataset = TensorDataset(X_test_t, y_test_t)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [24]:
class StockTransformer(nn.Module):
    def __init__(self, num_features, d_model=64, nhead=4, num_layers=2):
        super(StockTransformer, self).__init__()
        
        self.input_layer = nn.Linear(num_features, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            batch_first=True,
            dim_feedforward=128,
            dropout=0.1
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_layer = nn.Linear(d_model, 1)

    def forward(self, x):
        x = self.input_layer(x) 
        x = self.transformer_encoder(x) 
        x = x[:, -1, :] 
        out = self.output_layer(x)
        return out.squeeze()
num_features = X_train_seq.shape[2] 
model = StockTransformer(num_features=num_features)
print(model)

StockTransformer(
  (input_layer): Linear(in_features=12, out_features=64, bias=True)
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (output_layer): Linear(in_features=64, out_features=1, bias=True)
)


In [25]:
samples_train, seq_len, features = X_train_seq.shape
X_train_2d = X_train_seq.reshape(-1, features)

samples_test, _, _ = X_test_seq.shape
X_test_2d = X_test_seq.reshape(-1, features)

scaler = StandardScaler()
X_train_scaled_2d = scaler.fit_transform(X_train_2d)
X_test_scaled_2d = scaler.transform(X_test_2d)

X_train_seq_scaled = X_train_scaled_2d.reshape(samples_train, seq_len, features)
X_test_seq_scaled = X_test_scaled_2d.reshape(samples_test, seq_len, features)

In [28]:
X_train_t = torch.tensor(X_train_seq_scaled, dtype=torch.float32)
X_test_t = torch.tensor(X_test_seq_scaled, dtype=torch.float32)

y_train_seq_cls = (y_train_seq > 0).astype(np.float32)
y_test_seq_cls = (y_test_seq > 0).astype(np.float32)
y_train_t_cls = torch.tensor(y_train_seq_cls)
y_test_t_cls = torch.tensor(y_test_seq_cls)

train_dataset = TensorDataset(X_train_t, y_train_t_cls)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_dataset = TensorDataset(X_test_t, y_test_t_cls)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = StockTransformer(num_features=features)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

EPOCHS = 8

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    correct_preds = 0
    total_preds = 0
    
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_X).squeeze()
        targets = batch_y.float().squeeze()
        
        loss = criterion(predictions, targets)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        predicted_classes = (predictions > 0).float()
        correct_preds += (predicted_classes == targets).sum().item()
        total_preds += targets.size(0)
        
    avg_loss = epoch_loss / len(train_loader)
    epoch_acc = correct_preds / total_preds
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch + 1}/{EPOCHS}] | Accuracy: {epoch_acc:.4f}")

model.eval()
correct_preds_test = 0
total_preds_test = 0

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        predictions = model(batch_X).squeeze()
        targets = batch_y.float().squeeze()
        
        predicted_classes = (predictions > 0).float()
        
        correct_preds_test += (predicted_classes == targets).sum().item()
        total_preds_test += targets.size(0)

test_accuracy = correct_preds_test / total_preds_test

#I dont know how to set seed so the results are not reproducible but im getting 56% roughly, sometimes more sometimes less

print(f"Final Test Accuracy: {test_accuracy:.4f}")

Epoch [1/8] | Accuracy: 0.5066
Epoch [5/8] | Accuracy: 0.5516
Final Test Accuracy: 0.5340
